# 中证800 V72 V46排序质量诊断：topK recall / MAP / NDCG

独立版 notebook，可上传到 JoinQuant 研究环境运行。

目标：不改 V46-family 训练方式，只评估模型前排排序质量。重点回答：模型 top8 是否命中真实强势区，MAP/NDCG 是否高于随机，失败月是否对应 ranking 指标恶化。

固定训练协议：`full/direct/fixed120/alpha_1m/legacy_rebalance`。


## 0. 导入与进度条


In [ ]:
import os
import gc
import warnings
import builtins as _bi
from pathlib import Path

import lightgbm as lgb
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 260)
pd.set_option("display.width", 260)

try:
    from tqdm.auto import tqdm
except Exception:
    tqdm = None


def progress_iter(iterable, total=None, desc="progress", leave=True):
    if tqdm is not None:
        return tqdm(iterable, total=total, desc=desc, leave=leave)
    def _gen():
        every = _bi.max(1, int((total or 100) / 20))
        for i, item in enumerate(iterable, 1):
            if i == 1 or i % every == 0 or (total is not None and i == total):
                print("%s %s%s" % (desc, i, "/%s" % total if total else ""))
            yield item
    return _gen()


def display_df(df, n=30):
    try:
        display(df.head(n))
    except Exception:
        print(df.head(n).to_string(index=False))


## 1. 配置


In [ ]:
PROJECT_DIR = Path.cwd()
OUT_DIR = PROJECT_DIR / "csi800_ml_v72_ranking_quality_outputs"
OUT_DIR.mkdir(parents=True, exist_ok=True)

DATA_CANDIDATES = [
    Path("train_csi800_factor_v40_data_enhancement_20190101_20260531.csv"),
    Path("train_csi800_factor_v40_data_enhancement.csv"),
    Path("data/train_csi800_factor_v40_data_enhancement_20190101_20260531.csv"),
    Path("data/train_csi800_factor_v40_data_enhancement.csv"),
    PROJECT_DIR / "train_csi800_factor_v40_data_enhancement_20190101_20260531.csv",
    PROJECT_DIR / "train_csi800_factor_v40_data_enhancement.csv",
    PROJECT_DIR / "data" / "train_csi800_factor_v40_data_enhancement_20190101_20260531.csv",
    PROJECT_DIR / "data" / "train_csi800_factor_v40_data_enhancement.csv",
]
DATA_PATH_OVERRIDE = None

TARGET_COL = "alpha_1m"
STOCK_COL = "stock"
DATE_COL = "rebalance_date"
INDUSTRY_COL = "industry_bucket"
BENCHMARK = "000906.XSHG"

FIXED_ITER = 120
SEED = 42
CORR_THRESHOLD = 0.70
LABEL_BOUNDARY_MODE = "legacy_rebalance"

PREDICT_K_LIST = [8, 10, 15, 20]
TRUE_K_LIST = [8, 20, 50, 100]
NDCG_K_LIST = [8, 15, 20]
RANDOM_SIM_N = 500
RANDOM_SEED = 42

SAVE_SCORE_PANEL = True
SMOKE_TEST = False
SMOKE_MAX_MODELS = 1

MODEL_SPECS = [
    {"model_tag": "exp_2021_12", "train_start": "2019-01-01", "train_end": "2021-12-31", "test_start": "2022-01-01", "test_end": "2023-12-31", "phase": "early_36m"},
    {"model_tag": "exp_2022_12", "train_start": "2019-01-01", "train_end": "2022-12-31", "test_start": "2023-01-01", "test_end": "2024-12-31", "phase": "transition_48m"},
    {"model_tag": "exp_2023_12", "train_start": "2019-01-01", "train_end": "2023-12-31", "test_start": "2024-01-01", "test_end": "2025-12-31", "phase": "robust_ge60m"},
    {"model_tag": "exp_2024_12", "train_start": "2019-01-01", "train_end": "2024-12-31", "test_start": "2025-01-01", "test_end": "2026-04-30", "phase": "robust_ge60m"},
    {"model_tag": "exp_2025_12", "train_start": "2019-01-01", "train_end": "2025-12-31", "test_start": "2026-01-01", "test_end": "2026-06-30", "phase": "robust_ge60m"},
]
if SMOKE_TEST:
    MODEL_SPECS = MODEL_SPECS[:SMOKE_MAX_MODELS]

MANUAL_FAILURE_MONTHS = [
    "2022-04-01", "2022-08-01", "2023-09-01", "2024-01-02", "2024-03-01",
    "2025-03-03", "2025-05-06", "2025-11-03", "2026-03-02",
]

print("OUT_DIR:", OUT_DIR)
print("MODEL_SPECS:", [x["model_tag"] for x in MODEL_SPECS])


## 2. V46 full 特征与参数


In [ ]:
BASE_FACTOR_COLS = [
    "cash_flow_to_price_ratio", "book_to_price_ratio", "earnings_yield", "sales_to_price_ratio",
    "cash_earnings_to_price_ratio", "earnings_to_price_ratio", "roe_ttm", "roa_ttm",
    "gross_profit_ttm", "operating_profit_to_total_profit", "net_operate_cash_flow_to_total_liability",
    "net_operating_cash_flow_coverage", "adjusted_profit_to_total_profit", "ACCA", "growth",
    "net_working_capital", "operating_profit_per_share", "net_operate_cash_flow_per_share",
    "total_operating_revenue_per_share", "super_quick_ratio", "MLEV", "debt_to_equity_ratio",
    "debt_to_tangible_equity_ratio", "momentum", "Rank1M", "sharpe_ratio_60", "Variance20",
    "liquidity", "beta", "ATR6", "MFI14", "DAVOL10", "VOL10", "VMACD", "VOSC",
    "Skewness20", "Kurtosis20",
]

HYBRID_LIGHT_EXTRA_COLS = [
    "liq_money_ratio_20_60", "liq_paused_count_20", "px_close_to_ma60", "px_drawdown_60",
    "ts_cash_flow_to_price_ratio_rank_mean_3m", "ts_Rank1M_rank_chg_1m",
]

FULL_FEATURE_COLS = BASE_FACTOR_COLS + HYBRID_LIGHT_EXTRA_COLS

BASE_PARAMS_FF10 = {
    "objective": "regression",
    "metric": "l2",
    "boosting_type": "gbdt",
    "learning_rate": 0.05,
    "num_leaves": 31,
    "min_data_in_leaf": 200,
    "feature_fraction": 1.0,
    "bagging_fraction": 0.8,
    "bagging_freq": 1,
    "lambda_l1": 0.1,
    "lambda_l2": 0.3,
    "verbose": -1,
}


## 3. 训练/打分工具函数


In [ ]:
def unique_keep_order(cols):
    seen = set()
    out = []
    for col in cols:
        if col not in seen:
            out.append(col)
            seen.add(col)
    return out


def resolve_data_path():
    if DATA_PATH_OVERRIDE:
        p = Path(DATA_PATH_OVERRIDE)
        if p.exists():
            return p
        raise IOError("DATA_PATH_OVERRIDE not found: %s" % p)
    candidates = [Path(x) for x in DATA_CANDIDATES]
    for p in candidates:
        if p.exists():
            return p
    searched = [str(p.resolve()) for p in candidates]
    raise IOError("training data csv not found. Put train_csi800_factor_v40_data_enhancement*.csv in one of: %s" % searched)


def safe_to_datetime(df, cols):
    out = df.copy()
    for col in cols:
        if col in out.columns:
            out[col] = pd.to_datetime(out[col], errors="coerce").dt.normalize()
    return out


def safe_rank_ic(a, b):
    s = pd.DataFrame({"a": np.asarray(a, dtype=float), "b": np.asarray(b, dtype=float)})
    s = s.replace([np.inf, -np.inf], np.nan).dropna()
    if len(s) < 3 or s["a"].nunique() < 2 or s["b"].nunique() < 2:
        return np.nan
    return s["a"].rank(pct=True).corr(s["b"].rank(pct=True))


def build_corr_components(train_df, feature_cols, threshold):
    from collections import defaultdict
    corr = train_df[feature_cols].corr()
    graph = defaultdict(list)
    for i in range(len(feature_cols)):
        for j in range(i + 1, len(feature_cols)):
            v = corr.iloc[i, j]
            if not pd.isnull(v) and abs(v) > threshold:
                graph[feature_cols[i]].append(feature_cols[j])
                graph[feature_cols[j]].append(feature_cols[i])
    for col in feature_cols:
        graph[col]
    visited = set()
    comps = []

    def dfs(x, comp):
        visited.add(x)
        comp.append(x)
        for y in graph[x]:
            if y not in visited:
                dfs(y, comp)

    for col in feature_cols:
        if col not in visited:
            comp = []
            dfs(col, comp)
            comps.append(comp)
    return comps


def select_features_train_only(train_df, candidate_cols):
    cols = unique_keep_order([c for c in candidate_cols if c in train_df.columns])
    if len(cols) == 0:
        raise ValueError("no candidate feature exists in train data")
    missing = train_df[cols].isnull().sum().to_dict()
    keep = []
    remove = []
    for comp in build_corr_components(train_df, cols, CORR_THRESHOLD):
        if len(comp) == 1:
            keep.append(comp[0])
        else:
            comp = _bi.sorted(comp, key=lambda x: (missing[x], x))
            keep.append(comp[0])
            remove.extend(comp[1:])
    return keep, remove


def prepare_xy(df, feature_cols, target_col, fill_values=None):
    d = df.dropna(subset=[target_col]).copy()
    X = d.reindex(columns=feature_cols).replace([np.inf, -np.inf], np.nan).copy()
    y = d[target_col].astype(float).copy()
    if fill_values is None:
        fill_values = X.median().replace([np.inf, -np.inf], np.nan).fillna(0)
    X = X.fillna(fill_values).fillna(0)
    return X, y, fill_values, d.index


def load_dataset(path):
    df = pd.read_csv(path)
    df = safe_to_datetime(df, [DATE_COL, "feature_date", "next_date"])
    if STOCK_COL not in df.columns:
        for alt in ["code", "security", "order_book_id"]:
            if alt in df.columns:
                df = df.rename(columns={alt: STOCK_COL})
                break
    if TARGET_COL not in df.columns:
        if "raw_return_1m" in df.columns and "benchmark_csi800_1m" in df.columns:
            df[TARGET_COL] = df["raw_return_1m"] - df["benchmark_csi800_1m"]
        else:
            raise ValueError("target column not found: " + TARGET_COL)
    if INDUSTRY_COL not in df.columns:
        df[INDUSTRY_COL] = "UNKNOWN"
    need = [STOCK_COL, DATE_COL, TARGET_COL, INDUSTRY_COL, "feature_date", "next_date"]
    missing = [c for c in need if c not in df.columns]
    if missing:
        raise ValueError("dataset missing columns: " + ",".join(missing))
    df[TARGET_COL] = pd.to_numeric(df[TARGET_COL], errors="coerce")
    df[STOCK_COL] = df[STOCK_COL].astype(str)
    df = df.dropna(subset=[STOCK_COL, DATE_COL, TARGET_COL]).copy()
    return df


def make_train_df(df_all, spec):
    start = pd.Timestamp(spec["train_start"])
    end = pd.Timestamp(spec["train_end"])
    mask = (df_all[DATE_COL] >= start) & (df_all[DATE_COL] <= end)
    if LABEL_BOUNDARY_MODE == "label_end_safe":
        mask = mask & (df_all["next_date"] <= end)
    elif LABEL_BOUNDARY_MODE != "legacy_rebalance":
        raise ValueError("unknown LABEL_BOUNDARY_MODE: " + str(LABEL_BOUNDARY_MODE))
    return df_all[mask].copy()


def make_test_df(df_all, spec):
    start = pd.Timestamp(spec["test_start"])
    end = pd.Timestamp(spec["test_end"])
    return df_all[(df_all[DATE_COL] >= start) & (df_all[DATE_COL] <= end)].copy()


def train_direct_lgb(train_df, feature_cols):
    params = dict(BASE_PARAMS_FF10)
    params["seed"] = SEED
    X_train, y_train, fill_values, _ = prepare_xy(train_df, feature_cols, TARGET_COL)
    if len(X_train) == 0:
        raise ValueError("empty training matrix")
    model = lgb.train(params, lgb.Dataset(X_train, label=y_train), num_boost_round=_bi.max(1, int(FIXED_ITER)))
    pred = np.asarray(model.predict(X_train[feature_cols], num_iteration=FIXED_ITER)).reshape(-1)
    return {"model": model, "fill_values": fill_values, "train_rows": int(len(X_train)), "train_rank_ic": safe_rank_ic(y_train, pred)}


def score_with_model(df, model, feature_cols, fill_values):
    X = df.reindex(columns=feature_cols).replace([np.inf, -np.inf], np.nan).copy()
    X = X.fillna(fill_values).fillna(0)
    return np.asarray(model.predict(X[feature_cols], num_iteration=FIXED_ITER)).reshape(-1)


## 4. 排序指标函数


In [ ]:
def average_precision_at_k(pred_order, relevant_set, k):
    if len(relevant_set) == 0:
        return np.nan
    hits = 0
    score = 0.0
    top = pred_order[:_bi.min(k, len(pred_order))]
    for i, stock in enumerate(top, 1):
        if stock in relevant_set:
            hits += 1
            score += hits / float(i)
    denom = float(_bi.min(len(relevant_set), k))
    return score / denom if denom > 0 else np.nan


def dcg_at_k(gains, k):
    vals = np.asarray(gains[:_bi.min(k, len(gains))], dtype=float)
    if len(vals) == 0:
        return np.nan
    # Shift gains so negative alpha does not create inverted relevance in NDCG.
    vals = vals - np.nanmin(vals)
    denom = np.log2(np.arange(2, len(vals) + 2))
    return float(np.nansum(vals / denom))


def ndcg_at_k(pred_order, gain_map, k):
    pred_gains = [gain_map.get(s, np.nan) for s in pred_order]
    ideal_gains = _bi.sorted([v for v in gain_map.values() if not pd.isnull(v)], reverse=True)
    dcg = dcg_at_k(pred_gains, k)
    idcg = dcg_at_k(ideal_gains, k)
    if pd.isnull(dcg) or pd.isnull(idcg) or idcg <= 0:
        return np.nan
    return dcg / idcg


def random_ranking_metrics(stocks, true_sets, gain_map, pred_k_list, true_k_list, ndcg_k_list, n_sim=RANDOM_SIM_N, seed=RANDOM_SEED):
    rng = np.random.RandomState(seed)
    rows = []
    arr = np.arange(len(stocks))
    for _ in range(int(n_sim)):
        perm = rng.permutation(arr)
        pred_order = [stocks[i] for i in perm]
        row = {}
        for pk in pred_k_list:
            pred_set = set(pred_order[:_bi.min(pk, len(pred_order))])
            for tk in true_k_list:
                tset = true_sets.get(tk, set())
                row["precision_true_top%s_at%s" % (tk, pk)] = len(pred_set & tset) / float(pk) if pk > 0 else np.nan
                row["recall_true_top%s_at%s" % (tk, pk)] = len(pred_set & tset) / float(tk) if tk > 0 else np.nan
                row["map_true_top%s_at%s" % (tk, pk)] = average_precision_at_k(pred_order, tset, pk)
        for nk in ndcg_k_list:
            row["ndcg_alpha_at%s" % nk] = ndcg_at_k(pred_order, gain_map, nk)
        rows.append(row)
    return pd.DataFrame(rows)


def summarize_random_baseline(rand_df):
    out = {}
    for col in rand_df.columns:
        s = pd.to_numeric(rand_df[col], errors="coerce").dropna()
        if len(s) == 0:
            continue
        out[col + "_random_mean"] = float(s.mean())
        out[col + "_random_p50"] = float(s.quantile(0.50))
        out[col + "_random_p90"] = float(s.quantile(0.90))
    return out


## 5. 训练模型并生成低内存 score panel


In [ ]:
DATA_PATH = resolve_data_path()
df_all = load_dataset(DATA_PATH)
print("DATA_PATH:", DATA_PATH)
print("loaded:", df_all.shape)
print("rebalance_date:", df_all[DATE_COL].min(), "->", df_all[DATE_COL].max())

score_parts = []
model_meta_rows = []
for spec in progress_iter(MODEL_SPECS, total=len(MODEL_SPECS), desc="train V72 V46 models"):
    train_df = make_train_df(df_all, spec)
    test_df = make_test_df(df_all, spec)
    if train_df.empty or test_df.empty:
        print("skip empty", spec["model_tag"], train_df.shape, test_df.shape)
        continue
    feature_cols, removed_cols = select_features_train_only(train_df, FULL_FEATURE_COLS)
    trained = train_direct_lgb(train_df, feature_cols)
    pred = score_with_model(test_df, trained["model"], feature_cols, trained["fill_values"])
    keep_cols = [STOCK_COL, DATE_COL, TARGET_COL, INDUSTRY_COL, "next_date"]
    keep_cols = [c for c in keep_cols if c in test_df.columns]
    score_df = test_df[keep_cols].copy()
    score_df["score"] = pred
    score_df["model_tag"] = spec["model_tag"]
    score_df["phase"] = spec.get("phase", "")
    score_df["train_start"] = pd.Timestamp(spec["train_start"])
    score_df["train_end"] = pd.Timestamp(spec["train_end"])
    score_parts.append(score_df)
    model_meta_rows.append({
        "model_tag": spec["model_tag"],
        "phase": spec.get("phase", ""),
        "train_start": spec["train_start"],
        "train_end": spec["train_end"],
        "test_start": spec["test_start"],
        "test_end": spec["test_end"],
        "train_months": int(train_df[DATE_COL].nunique()),
        "train_rows": int(len(train_df)),
        "test_months": int(test_df[DATE_COL].nunique()),
        "test_rows": int(len(test_df)),
        "feature_count": int(len(feature_cols)),
        "removed_feature_count": int(len(removed_cols)),
        "train_rank_ic": trained["train_rank_ic"],
        "feature_cols": ",".join(feature_cols),
        "removed_features": ",".join(removed_cols),
    })
    del trained, train_df, test_df, score_df
    gc.collect()

score_panel_df = pd.concat(score_parts, ignore_index=True) if score_parts else pd.DataFrame()
model_meta_df = pd.DataFrame(model_meta_rows)
model_meta_df.to_csv(OUT_DIR / "v72_model_meta.csv", index=False)
if SAVE_SCORE_PANEL:
    score_panel_df.to_csv(OUT_DIR / "v72_score_panel.csv", index=False)
print("score_panel:", score_panel_df.shape)
display_df(model_meta_df, 20)


## 6. 计算 recall / MAP / NDCG 与随机基准


In [ ]:
metric_rows = []
rand_rows = []
failure_dates = set(pd.to_datetime(MANUAL_FAILURE_MONTHS).normalize())

grouped = score_panel_df.groupby(["model_tag", DATE_COL]) if len(score_panel_df) else []
ngroups = score_panel_df.groupby(["model_tag", DATE_COL]).ngroups if len(score_panel_df) else 0
for (model_tag, dt), gdf in progress_iter(grouped, total=ngroups, desc="ranking metrics"):
    m = gdf.dropna(subset=["score", TARGET_COL]).copy()
    if len(m) < _bi.max(TRUE_K_LIST + PREDICT_K_LIST):
        continue
    m[STOCK_COL] = m[STOCK_COL].astype(str)
    pred_order = m.sort_values("score", ascending=False)[STOCK_COL].tolist()
    true_order = m.sort_values(TARGET_COL, ascending=False)[STOCK_COL].tolist()
    stocks = m[STOCK_COL].tolist()
    gain_map = dict(zip(m[STOCK_COL], pd.to_numeric(m[TARGET_COL], errors="coerce")))
    true_sets = {}
    for tk in TRUE_K_LIST:
        true_sets[tk] = set(true_order[:_bi.min(tk, len(true_order))])

    rank_ic = safe_rank_ic(m["score"], m[TARGET_COL])
    row = {
        "model_tag": model_tag,
        "rebalance_date": pd.Timestamp(dt),
        "phase": str(m["phase"].iloc[0]) if "phase" in m.columns and len(m) else "",
        "n_universe": int(len(m)),
        "rank_ic": rank_ic,
        "manual_failure_month": bool(pd.Timestamp(dt).normalize() in failure_dates),
    }
    for pk in PREDICT_K_LIST:
        pred_top = pred_order[:_bi.min(pk, len(pred_order))]
        pred_set = set(pred_top)
        pred_ret = np.nanmean([gain_map.get(s, np.nan) for s in pred_top]) if len(pred_top) else np.nan
        row["pred_top%s_alpha" % pk] = float(pred_ret) if not pd.isnull(pred_ret) else np.nan
        for tk in TRUE_K_LIST:
            tset = true_sets[tk]
            row["precision_true_top%s_at%s" % (tk, pk)] = len(pred_set & tset) / float(pk)
            row["recall_true_top%s_at%s" % (tk, pk)] = len(pred_set & tset) / float(tk)
            row["hit_count_true_top%s_at%s" % (tk, pk)] = int(len(pred_set & tset))
            row["map_true_top%s_at%s" % (tk, pk)] = average_precision_at_k(pred_order, tset, pk)
    for nk in NDCG_K_LIST:
        row["ndcg_alpha_at%s" % nk] = ndcg_at_k(pred_order, gain_map, nk)

    rand_df = random_ranking_metrics(stocks, true_sets, gain_map, PREDICT_K_LIST, TRUE_K_LIST, NDCG_K_LIST)
    rand_summary = summarize_random_baseline(rand_df)
    for k, v in rand_summary.items():
        row[k] = v
    # random percentile for selected topK alpha.
    for pk in PREDICT_K_LIST:
        col = "pred_top%s_alpha" % pk
        rand_alphas = []
        rng = np.random.RandomState(RANDOM_SEED)
        arr = np.arange(len(stocks))
        for _ in range(int(RANDOM_SIM_N)):
            perm = rng.permutation(arr)
            picked = [stocks[i] for i in perm[:_bi.min(pk, len(stocks))]]
            rand_alphas.append(np.nanmean([gain_map.get(s, np.nan) for s in picked]))
        rv = np.asarray(rand_alphas, dtype=float)
        row[col + "_random_mean"] = float(np.nanmean(rv))
        row[col + "_random_p50"] = float(np.nanpercentile(rv, 50))
        row[col + "_random_p90"] = float(np.nanpercentile(rv, 90))
        row[col + "_random_percentile"] = float((rv <= row[col]).mean()) if not pd.isnull(row[col]) else np.nan
    metric_rows.append(row)

metrics_monthly_df = pd.DataFrame(metric_rows)
metrics_monthly_df.to_csv(OUT_DIR / "v72_ranking_metrics_monthly.csv", index=False)
print("monthly metrics:", metrics_monthly_df.shape)
display_df(metrics_monthly_df, 10)


## 7. 汇总：模型窗口、成熟阶段、失败月


In [ ]:
def calc_nav(ret_series):
    s = pd.Series(ret_series).replace([np.inf, -np.inf], np.nan).fillna(0.0)
    if len(s) == 0:
        return pd.Series(dtype=float)
    return (1.0 + s).cumprod()


def calc_mdd(ret_series):
    nav = calc_nav(ret_series)
    if len(nav) == 0:
        return np.nan
    return float((nav / nav.cummax() - 1.0).min())


def summarize_metrics(df, keys):
    rows = []
    if len(df) == 0:
        return pd.DataFrame()
    metric_cols = [c for c in df.columns if c not in keys + ["rebalance_date", "phase", "manual_failure_month"]]
    for name, gdf in df.groupby(keys):
        if not isinstance(name, tuple):
            name = (name,)
        row = {}
        for i, key in enumerate(keys):
            row[key] = name[i]
        row["months"] = int(len(gdf))
        for col in metric_cols:
            if col.startswith("pred_top") and col.endswith("_alpha"):
                s = pd.to_numeric(gdf[col], errors="coerce").dropna()
                row[col + "_cum"] = float((1.0 + s).prod() - 1.0) if len(s) else np.nan
                row[col + "_mean"] = float(s.mean()) if len(s) else np.nan
                row[col + "_win_rate"] = float((s > 0).mean()) if len(s) else np.nan
                row[col + "_mdd"] = calc_mdd(s)
                row[col + "_worst"] = float(s.min()) if len(s) else np.nan
            elif col.endswith("_random_percentile") or col.startswith("precision_") or col.startswith("recall_") or col.startswith("map_") or col.startswith("ndcg_") or col == "rank_ic":
                s = pd.to_numeric(gdf[col], errors="coerce").dropna()
                row[col + "_mean"] = float(s.mean()) if len(s) else np.nan
                row[col + "_median"] = float(s.median()) if len(s) else np.nan
                row[col + "_positive_rate"] = float((s > 0).mean()) if len(s) else np.nan
        rows.append(row)
    return pd.DataFrame(rows)

summary_by_model_df = summarize_metrics(metrics_monthly_df, ["model_tag"])
robust_df = metrics_monthly_df[metrics_monthly_df["model_tag"].isin(["exp_2023_12", "exp_2024_12", "exp_2025_12"])].copy() if len(metrics_monthly_df) else pd.DataFrame()
robust_summary_df = summarize_metrics(robust_df.assign(group="robust_ge60m"), ["group"])
failure_df = metrics_monthly_df[metrics_monthly_df["manual_failure_month"] == True].copy() if len(metrics_monthly_df) else pd.DataFrame()
failure_summary_df = summarize_metrics(failure_df.assign(group="manual_failure_months"), ["group"])

summary_by_model_df.to_csv(OUT_DIR / "v72_ranking_summary_by_model.csv", index=False)
robust_summary_df.to_csv(OUT_DIR / "v72_ranking_summary_robust_phase.csv", index=False)
failure_summary_df.to_csv(OUT_DIR / "v72_ranking_summary_failure_months.csv", index=False)

# Compact view for key metrics.
key_cols = [
    "model_tag", "months", "rank_ic_mean",
    "pred_top8_alpha_cum", "pred_top8_alpha_mean", "pred_top8_alpha_random_percentile_mean",
    "precision_true_top20_at8_mean", "precision_true_top50_at8_mean", "precision_true_top100_at8_mean",
    "recall_true_top20_at8_mean", "map_true_top20_at8_mean", "ndcg_alpha_at8_mean",
]
key_cols = [c for c in key_cols if c in summary_by_model_df.columns]
display_df(summary_by_model_df[key_cols], 20)

robust_key_cols = [c for c in key_cols if c != "model_tag"]
robust_key_cols = ["group"] + [c for c in robust_key_cols if c in robust_summary_df.columns]
display_df(robust_summary_df[robust_key_cols], 10)
display_df(failure_summary_df[[c for c in robust_key_cols if c in failure_summary_df.columns]], 10)


## 8. 随机基准对比与健康监测候选


In [ ]:
comparison_rows = []
if len(metrics_monthly_df):
    compare_cols = []
    for base in [
        "precision_true_top20_at8", "precision_true_top50_at8", "precision_true_top100_at8",
        "map_true_top20_at8", "map_true_top50_at8", "ndcg_alpha_at8", "pred_top8_alpha",
    ]:
        if base in metrics_monthly_df.columns:
            compare_cols.append(base)
    groups = [("all", metrics_monthly_df), ("robust_ge60m", robust_df), ("failure_months", failure_df)]
    for gname, gdf in groups:
        if len(gdf) == 0:
            continue
        row = {"group": gname, "months": int(len(gdf))}
        for base in compare_cols:
            s = pd.to_numeric(gdf[base], errors="coerce")
            row[base + "_mean"] = float(s.mean())
            for suffix in ["_random_mean", "_random_p50", "_random_p90"]:
                rcol = base + suffix
                if rcol in gdf.columns:
                    rs = pd.to_numeric(gdf[rcol], errors="coerce")
                    row[base + suffix + "_mean"] = float(rs.mean())
                    row[base + "_edge_vs" + suffix.replace("_random", "random")] = float(s.mean() - rs.mean())
            if base + "_random_percentile" in gdf.columns:
                ps = pd.to_numeric(gdf[base + "_random_percentile"], errors="coerce")
                row[base + "_random_percentile_mean"] = float(ps.mean())
                row[base + "_random_percentile_lt25_rate"] = float((ps < 0.25).mean())
        comparison_rows.append(row)

comparison_df = pd.DataFrame(comparison_rows)
comparison_df.to_csv(OUT_DIR / "v72_random_baseline_comparison.csv", index=False)
display_df(comparison_df, 20)

# Health candidate: weak ranking months.
health_rows = []
if len(metrics_monthly_df):
    df = metrics_monthly_df.copy()
    for col in ["precision_true_top50_at8", "map_true_top20_at8", "ndcg_alpha_at8", "pred_top8_alpha_random_percentile"]:
        if col in df.columns:
            q25 = pd.to_numeric(df[col], errors="coerce").quantile(0.25)
            df[col + "_weak"] = pd.to_numeric(df[col], errors="coerce") <= q25
    weak_cols = [c for c in df.columns if c.endswith("_weak")]
    if weak_cols:
        df["ranking_weak_score"] = df[weak_cols].sum(axis=1)
        df["ranking_health_state"] = np.where(df["ranking_weak_score"] >= 3, "red", np.where(df["ranking_weak_score"] >= 2, "yellow", "green"))
        health_rows = df[["model_tag", "rebalance_date", "phase", "manual_failure_month", "ranking_weak_score", "ranking_health_state"] + [c for c in ["precision_true_top50_at8", "map_true_top20_at8", "ndcg_alpha_at8", "pred_top8_alpha_random_percentile", "pred_top8_alpha"] if c in df.columns]].to_dict("records")
health_df = pd.DataFrame(health_rows)
health_df.to_csv(OUT_DIR / "v72_ranking_health_state.csv", index=False)
if len(health_df):
    display_df(health_df.sort_values(["ranking_weak_score", "rebalance_date"], ascending=[False, True]), 30)

print("saved outputs:")
for fp in _bi.sorted(OUT_DIR.glob("v72_*.csv")):
    print("-", fp)
